<a href="https://colab.research.google.com/github/urazovandriy-dotcom/ML_LABS_2026/blob/main/%D0%9B%D0%A09_%D0%A3%D1%80%D0%B0%D0%B7%D0%BE%D0%B2_3_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Уразов Андрій Ярославович  
**Група:** 3-10  
**Дата:**  04.11.2026

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.16
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
print('Унікальних користувачів:', ratings_df['user_id'].nunique())
print('Унікальних фільмів:', ratings_df['item_id'].nunique())
print('Min rating:', ratings_df['rating'].min())
print('Max rating:', ratings_df['rating'].max())
print('Mean rating:', ratings_df['rating'].mean())
print('Пропуски:\n', ratings_df.isnull().sum())
print('\nРозподіл оцінок:\n', ratings_df['rating'].value_counts().sort_index())

Унікальних користувачів: 943
Унікальних фільмів: 1682
Min rating: 1.0
Max rating: 5.0
Mean rating: 3.52986
Пропуски:
 user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

Розподіл оцінок:
 rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

_Датасет містить 100 000 оцінок від 943 унікальних користувачів для 1682 фільмів без жодних пропущених значень. Оцінки варіюються в діапазоні від 1.0 до 5.0 із середнім значенням 3.53. Розподіл має помітний зсув у бік позитивного сприйняття: найбільшу частку становлять оцінки 4.0 (34 174) та 3.0 (27 145), тоді як найнижчий бал 1.0 зустрічається найрідше (лише 6 110 разів)._

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
train_df, test_df = sklearn_train_test_split(ratings_df, test_size=0.2, random_state=42)
print('train_df shape:', train_df.shape)
print('test_df shape:', test_df.shape)

train_df shape: (80000, 4)
test_df shape: (20000, 4)


In [6]:
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))
print('train_data instances:', len(train_df))
print('testset size:', len(testset))

train_data instances: 80000
testset size: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [8]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}
param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_SVD: кількість факторів n_factors (50, 100) балансує між деталізацією прихованих уподобань і захистом від перенавчання; lr_all (0.005, 0.010) забезпечує стабільну збіжність градієнтного спуску; reg_all (0.02, 0.05) контролює L2-регуляризацію для запобігання росту ваг.

KNNBasic: кількість сусідів k (20, 40) та min_k (1, 5) оптимізують згладжування шуму та фільтрують випадки з малим числом спільних оцінок; у sim_options порівнюються метрики схожості (cosine, pearson) та підходи item-item (user_based: False) проти user-user (user_based: True) для вибору найбільш стабільної конфігурації на розрідженій матриці._

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [9]:
grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_svd.fit(train_data)

In [10]:
grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_knn.fit(train_data)

In [11]:
print('SVD RMSE:', grid_search_svd.best_score['rmse'])
print('SVD MAE:', grid_search_svd.best_score['mae'])
print('SVD params:', grid_search_svd.best_params['rmse'])
print('\nKNNBasic RMSE:', grid_search_knn.best_score['rmse'])
print('KNNBasic MAE:', grid_search_knn.best_score['mae'])
print('KNNBasic params:', grid_search_knn.best_params['rmse'])

SVD RMSE: 0.9429200864622178
SVD MAE: 0.7440295645024803
SVD params: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

KNNBasic RMSE: 1.0272322697669731
KNNBasic MAE: 0.8149647455639611
KNNBasic params: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9429200864622178 | 0.7440295645024803 | {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05} |
| KNNBasic | 1.0272322697669731 | 0.8149647455639611 | {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}} |

**Порівняння моделей:**

Модель матричної факторизації SVD виявилася точнішою за обома метриками (RMSE 0.9429 проти 1.0272 та MAE 0.7440 проти 0.8150 у KNNBasic). Оптимальною для SVD стала конфігурація з 50 латентними факторами, швидкістю навчання 0.01 та регуляризацією 0.05. Для KNNBasic найкращі показники забезпечив підхід user-user колаборативної фільтрації (user_based: True) з косинусною мірою схожості (cosine), 40 сусідами та min_k = 1.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [13]:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']
trainset = train_data.build_full_trainset()
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)
best_svd.fit(trainset)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [14]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)
print(f"SVD Test -> RMSE: {svd_rmse:.4f}, MAE: {svd_mae:.4f}")
print(f"KNNBasic Test -> RMSE: {knn_rmse:.4f}, MAE: {knn_mae:.4f}")

SVD Test -> RMSE: 0.9192, MAE: 0.7234
KNNBasic Test -> RMSE: 1.0120, MAE: 0.7984


In [15]:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values(by='RMSE').reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:**

_Оцінки помилок на незалежній тестовій вибірці суттєво не відхиляються від результатів крос-валідації та демонструють аналогічну динаміку, будучи навіть дещо кращими: для SVD значення RMSE знизилося з $0.9429$ на CV до $0.9192$ на тесті (MAE з $0.7440$ до $0.7234$), а для KNNBasic RMSE склало $1.0120$ на тесті проти $1.0272$ на CV (MAE $0.7984$ проти $0.8150$). Така узгодженість підтверджує відсутність перенавчання (overfitting), а модель SVD впевнено підтвердила свою перевагу над підходом найближчих сусідів на обох етапах перевірки._

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [20]:
user_id = '196'
best_rec_model = best_svd if grid_search_svd.best_score['rmse'] < grid_search_knn.best_score['rmse'] else best_knn
if not trainset.knows_user(trainset.to_inner_uid(user_id) if user_id in [trainset.to_raw_uid(u) for u in trainset.all_users()] else -1):
    print(f"Користувача з ID {user_id} немає в навчальній вибірці!")
else:
    inner_uid = trainset.to_inner_uid(user_id)
    rated_inner_iids = {item_id for (item_id, _) in trainset.ur[inner_uid]}
    unrated_raw_iids = [trainset.to_raw_iid(i) for i in trainset.all_items() if i not in rated_inner_iids]
    predictions = [
        (raw_iid, best_rec_model.predict(uid=user_id, iid=raw_iid).est)
        for raw_iid in unrated_raw_iids
    ]
    top_10 = sorted(predictions, key=lambda x: x[1], reverse=True)[:10]
    top_10_df = pd.DataFrame(top_10, columns=['item_id', 'predicted_rating'])
    display(top_10_df)

,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

_За результатами крос-валідації найкращою моделлю обрано SVD, оскільки вона досягла відчутно меншої середньоквадратичної помилки $\text{CV RMSE} = 0.9429$ та середньої абсолютної похибки $\text{CV MAE} = 0.7440$ проти $1.0272$ та $0.8150$ у KNNBasic. Оптимальними гіперпараметрами для алгоритму SVD стали n_factors: 50, lr_all: 0.01 та reg_all: 0.05, а для моделі KNNBasic найкращою конфігурацією виявився підхід user-based із косинусною мірою схожості, k: 40 та min_k: 1. На незалежній тестовій вибірці SVD підтвердила лідерство із показниками $\text{RMSE} = 0.9192$ та $\text{MAE} = 0.7234$, тоді як у KNNBasic тестові похибки склали $\text{RMSE} = 1.0120$ та $\text{MAE} = 0.7984$. Відхилення між оцінками на крос-валідації та незалежному тесті є незначним, а тестові помилки навіть дещо нижчі, що свідчить про стабільність налаштованих моделей і відсутність ефекту перенавчання. Сформований топ-10 рекомендацій для користувача '196' виключив усі переглянуті ним раніше фільми, прогнозуючи найвищий персоналізований бал. Водночас обидва підходи мають класичне обмеження «холодного старту» (cold start), адже вони не здатні генерувати якісні прогнози для нових користувачів або нещодавно доданих фільмів без попередньої історії взаємодій. Окрім того, KNNBasic поступається у масштабованості через необхідність розрахунку попарної матриці схожості, тоді як факторизація матриць SVD працює набагато швидше й краще вловлює приховані латентні фактори переваг._

---
